# AgentCore Evaluation Pipeline

This notebook demonstrates a CI/CD-style evaluation pipeline for an AgentCore agent.
It deploys the stack, invokes the agent with test prompts using an **M2M token**
(machine-to-machine, no user identity), runs built-in evaluators, and checks
quality gates.

### Pipeline steps
1. **Deploy**: CDK creates Cognito, Agent Runtime, MCP Server Runtime
2. **Authenticate**: Get an M2M token via `client_credentials` grant
3. **Invoke**: Send test prompts to the agent (M2M bypasses role checks, all tools accessible)
4. **Evaluate**: Collect OTel traces and score with built-in evaluators
5. **Quality gate**: Fail if any score is below threshold

### Prerequisites
- AWS account with Bedrock AgentCore access
- AWS credentials configured (`aws configure`)
- Docker installed and running
- Node.js 18+ (required by CDK)
- CDK bootstrapped (`npx cdk bootstrap`)

## Step 0: Install Dependencies

If Node.js is not installed, uncomment the line for your OS:

In [ ]:
# --- Uncomment ONE line matching your OS to install Node.js ---

# macOS (with Homebrew)
#!brew install node

# macOS (without Homebrew) — installs to ~/.local, no sudo needed
#!mkdir -p ~/.local && curl -fsSL https://nodejs.org/dist/v18.20.8/node-v18.20.8-darwin-arm64.tar.xz | tar -xJ --strip-components=1 -C ~/.local && export PATH="$HOME/.local/bin:$PATH"

# Ubuntu / Debian
#!curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash - && sudo apt-get install -y nodejs

# Amazon Linux / RHEL
#!curl -fsSL https://rpm.nodesource.com/setup_18.x | sudo bash - && sudo yum install -y nodejs

# Windows (run in PowerShell)
#!winget install OpenJS.NodeJS.LTS

In [ ]:
# Install Python dependencies (CDK library, boto3, requests)
!cd .. && python3 -m venv .venv && source .venv/bin/activate && pip install -q .

## Step 1: Deploy the CDK Stack

Creates all AWS resources and writes connection details to `outputs.json`.

In [ ]:
# Deploy the stack — PATH includes /usr/local/bin so CDK can find Docker
!cd .. && PATH=$PWD/.venv/bin:$HOME/.local/bin:/usr/local/bin:/opt/homebrew/bin:/usr/bin:$PATH npx --yes cdk deploy --outputs-file outputs.json --require-approval never

## Step 2: Load Configuration

Read stack outputs and fetch the M2M client secret from Secrets Manager.

In [ ]:
import json, boto3

# Load CDK outputs
with open('../outputs.json') as f:
    outputs = json.load(f)['AgentCoreCICDStack-dev']

REGION = 'ap-southeast-2'
AGENT_RUNTIME_ARN = outputs['AgentRuntimeArn']
AGENT_RUNTIME_ID = outputs['AgentRuntimeId']
TOKEN_ENDPOINT = outputs['TokenEndpoint']
EVAL_THRESHOLD = 0.8

# Fetch M2M client credentials from Secrets Manager
# (CDK stores the Cognito client_id + client_secret here)
sm = boto3.client('secretsmanager', region_name=REGION)
secret = json.loads(sm.get_secret_value(SecretId='agentcore/dev/m2m-client')['SecretString'])
OAUTH_CLIENT_ID = secret['client_id']
OAUTH_CLIENT_SECRET = secret['client_secret']

print(f'Agent ARN: {AGENT_RUNTIME_ARN}')
print(f'Agent ID:  {AGENT_RUNTIME_ID}')
print(f'Token URL: {TOKEN_ENDPOINT}')

## Step 3: Get M2M Token

CI pipelines authenticate using the `client_credentials` OAuth grant.
This produces a token with scopes but no user roles — the MCP server's
`AuthMiddleware` detects this and bypasses role checks, giving access to all tools.

In [ ]:
import requests as http_requests

def get_m2m_token():
    """Get M2M token via client_credentials grant."""
    resp = http_requests.post(
        TOKEN_ENDPOINT,
        data={
            'grant_type': 'client_credentials',
            'client_id': OAUTH_CLIENT_ID,
            'client_secret': OAUTH_CLIENT_SECRET,
            'scope': 'agentcore/invoke',
        },
    )
    resp.raise_for_status()
    return resp.json()['access_token']

token = get_m2m_token()
print(f'Token acquired (first 20 chars): {token[:20]}...')

## Step 4: Invoke the Agent

OAuth-protected runtimes must be invoked via HTTPS with a Bearer token.
The boto3 SDK doesn't support OAuth invocations, so we use `requests` directly.

All prompts share a single session ID — this groups the OTel traces for evaluation.

In [ ]:
import time, uuid, urllib.parse

def invoke_agent(prompt, session_id, token):
    """Invoke AgentCore Runtime via HTTPS with Bearer token."""
    escaped_arn = urllib.parse.quote(AGENT_RUNTIME_ARN, safe='')
    url = f'https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{escaped_arn}/invocations?qualifier=DEFAULT'
    resp = http_requests.post(
        url,
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
        },
        data=json.dumps({'prompt': prompt}),
    )
    resp.raise_for_status()
    return resp.json()

# Test prompts — M2M token has access to ALL tools (role checks bypassed)
prompts = [
    'How much is 2+2?',                          # calculator (built-in)
    'What is the current time in UTC?',           # get_current_datetime (MCP, public)
    'What is the stock price of AAPL?',           # get_stock_price (MCP, FinanceUser — M2M bypasses)
    'How many employees are in engineering?',     # get_employee_count (MCP, HRUser — M2M bypasses)
]

# Use a single session so all traces are grouped for evaluation
session_id = str(uuid.uuid4())
print(f'Session ID: {session_id}\n')

for prompt in prompts:
    print(f'Q: {prompt}')
    try:
        result = invoke_agent(prompt, session_id, token)
        print(f'A: {result}\n')
    except Exception as e:
        print(f'Error: {e}\n')

## Step 5: Run Evaluations

The `bedrock-agentcore-starter-toolkit` collects OTel traces from the session
and scores them with built-in evaluators.

> **Note:** Traces take ~30s to propagate to CloudWatch after invocation.

In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation

evaluators = [
    'Builtin.GoalSuccessRate',
    'Builtin.Correctness',
    'Builtin.ToolSelectionAccuracy',
    'Builtin.ToolParameterAccuracy',
]

MAX_RETRIES = 4
WAIT_SECONDS = 30
results = None

for attempt in range(1, MAX_RETRIES + 1):
    print(f'Attempt {attempt}/{MAX_RETRIES}: waiting {WAIT_SECONDS}s for trace propagation...')
    time.sleep(WAIT_SECONDS)
    try:
        results = Evaluation(region=REGION).run(
            agent_id=AGENT_RUNTIME_ID,
            session_id=session_id,
            evaluators=evaluators,
            output='eval_output.json',
        )
        break
    except RuntimeError as e:
        if 'No spans found' in str(e) and attempt < MAX_RETRIES:
            print(f'  No spans yet, retrying...')
            continue
        raise

# Aggregate: keep best score per evaluator
scores = {}
for r in results.results:
    if r.value is not None:
        if r.evaluator_name not in scores or r.value > scores[r.evaluator_name][0]:
            scores[r.evaluator_name] = (r.value, r.label)

# Display results table
print(f'{chr(9472) * 50}')
print(f'{"Evaluator":<35} {"Score":>6}  Result')
print(f'{chr(9472) * 50}')
for name in evaluators:
    if name in scores:
        val, label = scores[name]
        icon = chr(9989) if val >= EVAL_THRESHOLD else chr(10060)
        print(f'{icon} {name:<33} {val:>5.1f}  {label}')
    else:
        print(f'{chr(9888)}{chr(65039)}  {name:<33}     -  no data')
print(f'{chr(9472) * 50}')

## Step 6: Quality Gate

In CI, this step exits non-zero to block the PR if any metric is below threshold.

In [ ]:
failed = any(val < EVAL_THRESHOLD for val, _ in scores.values())
if failed:
    print(f'\u274c FAILED: one or more metrics below {EVAL_THRESHOLD}')
else:
    print(f'\u2705 All evaluations PASSED (threshold: {EVAL_THRESHOLD})')

## How Role-Based Access Works

The MCP server enforces access control at the tool level:

1. **AgentCore** validates the JWT (signature, issuer, expiry)
2. **`request_header_allowlist=["Authorization"]`** passes the token through to the MCP container
3. **`AuthMiddleware`** reads `custom:roles` from the JWT and compares against tool metadata:

```python
@mcp.tool(meta=auth_meta(roles=['FinanceUser']))
def get_stock_price(symbol: str) -> str: ...

@mcp.tool(meta=auth_meta(roles=['HRUser']))
def get_employee_count(department: str) -> str: ...
```

- **M2M tokens** (scopes but no roles) → bypass role checks → all tools visible
- **User tokens** (`custom:roles` present) → only matching tools visible

## Cleanup

Destroy the CDK stack to avoid ongoing charges.

In [ ]:
!cd .. && PATH=$PWD/.venv/bin:$HOME/.local/bin:/usr/local/bin:/opt/homebrew/bin:/usr/bin:$PATH npx --yes cdk destroy --force